In [10]:
import math
from collections import defaultdict, Counter
class BeginnerKneserNey:
    def __init__(self, order=5, discount=0.75):
        self.order = order
        self.discount = discount
        self.vocab = set()
        self.unk = "<UNK>"
        self.sos = "<s>"
        self.eos = "</s>"
        
        # self.ngram_counts[n] will be a dictionary storing counts of tuples of size 'n'
        self.ngram_counts = [defaultdict(int) for _ in range(self.order + 1)]
        
        # Suffix and context counting structures for Continuation Probabilities
        # history_extensions[suffix] = set of unique words that preceded this suffix
        self.history_extensions = defaultdict(set)
        
        # context_extensions[context] = set of unique words that followed this context
        self.context_extensions = defaultdict(set)
        
        # Precomputed denominator values for lower-order context tiers
        # lower_order_denominators[context] = sum of continuation counts of all possible extensions
        self.lower_order_denominators = defaultdict(int)
        
        # Total number of unique bigrams in training data (used as unigram base-case denominator)
        self.total_unigram_continuations = 0
    def tokenize(self, text: str):
        words = text.lower().strip().split()
        return [self.sos] * (self.order - 1) + words + [self.eos]

    def fit(self, corpus: list):
   
        # Step 1: Build the vocabulary using words appearing in the corpus
        raw_word_frequencies = Counter()
        for sentence in corpus:
            tokens = sentence.lower().strip().split()
            raw_word_frequencies.update(tokens)
        # We only keep words appearing > 1 times to keep the model fast and compact
        self.vocab = {word for word, count in raw_word_frequencies.items() if count > 1}
        self.vocab.add(self.unk)
        self.vocab.add(self.sos)
        self.vocab.add(self.eos)
        # Step 2: Convert unseen words in the sentences to <UNK>
        processed_sentences = []
        for sentence in corpus:
            tokens = sentence.lower().strip().split()
            clean_tokens = [self.sos] * (self.order - 1) + \
                [t if t in self.vocab else self.unk for t in tokens] + \
                [self.eos]
            processed_sentences.append(clean_tokens)
        # Step 3: Count n-grams using a sliding window for all order tiers [1 to order]
        for tokens in processed_sentences:
            for n in range(1, self.order + 1):
                for i in range(len(tokens) - n + 1):
                    ngram = tuple(tokens[i : i + n])
                    self.ngram_counts[n][ngram] += 1
                    
                    if n > 1:
                        history = ngram[:-1]
                        target_word = ngram[-1]
                        preceding_word = ngram[0]
                        suffix_ngram = ngram[1:]
                        self.history_extensions[suffix_ngram].add(preceding_word)
                        self.context_extensions[history].add(target_word)
                    
        # Step 4: Precompute unigram continuation normalizer (total unique bigrams seen)
        all_unique_bigrams = set(self.ngram_counts[2].keys())
        self.total_unigram_continuations = len(all_unique_bigrams)

        # Step 5: Precompute lower-order denominators for absolute probability normalization
        for suffix_ngram, preceding_words in self.history_extensions.items():
            history_context = suffix_ngram[:-1]
            self.lower_order_denominators[history_context] += len(preceding_words)

    def _get_continuation_count(self, ngram: tuple):
        return len(self.history_extensions[ngram])

    def _get_context_count(self, context: tuple):
        return len(self.context_extensions[context])

    def get_probability(self, word: str, context: tuple) -> float:
        if len(context) >= self.order:
            context = context[-(self.order - 1):]
        word = word if word in self.vocab else self.unk
        context = tuple(w if w in self.vocab else self.unk for w in context)
    
        n = len(context) + 1
        full_ngram = context + (word,)

        if n == 1:
            num = self._get_continuation_count((word,))
            den = self.total_unigram_continuations
            return num / den if den > 0 else 1.0 / len(self.vocab)
    
        if n == self.order:
            count_numerator = self.ngram_counts[n][full_ngram]
            count_denominator = self.ngram_counts[n - 1][context]
        else:
            count_numerator = self._get_continuation_count(full_ngram)
            count_denominator = self.lower_order_denominators[context]
        
        if count_denominator > 0:
            discounted_term = max(count_numerator - self.discount, 0) / count_denominator
            num_extensions = self._get_context_count(context)
            interpolation_lambda = (self.discount / count_denominator) * num_extensions
        else:
            discounted_term = 0.0
            interpolation_lambda = 1.0
        lower_order_prob = self.get_probability(word, context[1:])
        return discounted_term + (interpolation_lambda * lower_order_prob)

    def calculate_sentence_perplexity(self, sentence: str) -> float:
        tokens = self.tokenize(sentence)
        total_log_prob = 0.0
        n_words = 0
        for i in range(self.order - 1, len(tokens)):
            word = tokens[i]
            context = tuple(tokens[i - (self.order - 1) : i])
            prob = self.get_probability(word, context)
            prob = max(prob, 1e-12)
            total_log_prob += math.log2(prob)
            n_words += 1

        if n_words == 0:
            return float('inf')
        cross_entropy = -total_log_prob / n_words
        return 2 ** cross_entropy

if __name__ == "__main__":
    toy_corpus = [
        "The quick brown fox jumps over the lazy dog",
        "A quick brown fox running along the river bank",
        "The dog barked loudly at the quick brown fox",
        "Never run near the river bank in the dark",
        "The quick brown fox leaped high in the air"
      ]
        
    print("Initializing 5-gram Kneser-Ney Language Model...")
    lm = BeginnerKneserNey(order=5, discount=0.75)
    lm.fit(toy_corpus)

    print(f"Total Model Vocabulary Size: {len(lm.vocab)}")

    test_context = ("the", "quick", "brown", "fox")
    prob_sum = 0.0
    for vocab_word in lm.vocab:
        prob_sum += lm.get_probability(vocab_word, test_context)

    print(f"Sum of P(w | 'the quick brown fox') over all vocab: {prob_sum:.6f} (Expected: ~1.0)")
        
    word_candidates = ["jumps", "leaped", "barked", "river"]
    print("\nConditional Likelihood Predictions for context 'the quick brown fox':")
    for w in word_candidates:
        p = lm.get_probability(w, test_context)
        print(f" - P({w:<10} | context) = {p:.6f}")

    test_sentence = "The quick brown fox jumps"
    perplexity = lm.calculate_sentence_perplexity(test_sentence)
    print(f"\nSentence: '{test_sentence}' -> Perplexity: {perplexity:.4f}")
        
    out_of_vocab_test = "The fast magenta unicorn jumps"
    perplexity_oov = lm.calculate_sentence_perplexity(out_of_vocab_test)
    print(f"Sentence: '{out_of_vocab_test}' (contains OOVs) -> Perplexity: {perplexity_oov:.4f}")

Initializing 5-gram Kneser-Ney Language Model...
Total Model Vocabulary Size: 11
Sum of P(w | 'the quick brown fox') over all vocab: 1.000000 (Expected: ~1.0)

Conditional Likelihood Predictions for context 'the quick brown fox':
 - P(jumps      | context) = 0.710258
 - P(leaped     | context) = 0.710258
 - P(barked     | context) = 0.710258
 - P(river      | context) = 0.006114

Sentence: 'The quick brown fox jumps' -> Perplexity: 2.5597
Sentence: 'The fast magenta unicorn jumps' (contains OOVs) -> Perplexity: 5.7640
